In [ ]:
import os

PHYSICAL_CORES = 56
N_THREADS = 112

os.environ["POLARS_MAX_THREADS"] = str(N_THREADS)
os.environ["MKL_NUM_THREADS"] = str(PHYSICAL_CORES)
os.environ["OPENBLAS_NUM_THREADS"] = str(PHYSICAL_CORES)
os.environ["OMP_NUM_THREADS"] = str(PHYSICAL_CORES) # for faiss

import sys
#!pip install polars --target ./my_custom_packages
#sys.path.append("./my_custom_packages") #  faiss-cpu
custom_path = "/stor/home/he4249/orange/my_custom_packages" 
sys.path.insert(0, custom_path)

import polars as pl
import pandas as pd
import faiss
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
# Umap needs numpy doesn't support numpy 2.5+
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import time
import matplotlib.pyplot as plt



pl.Config.set_tbl_rows(50)
pl.Config(tbl_cols = 50)


In [ ]:
t1 = time.time()

df_merged = pl.read_parquet("df_merged.parquet")
df_merged = df_merged.with_columns(
    pl.col("CA").log1p().alias("log_ca"),
    pl.col("nb_jours").log1p().alias("lognbj")
)
df_merged_pd = df_merged.to_pandas().set_index("msisdn")
#display(df_merged.head())s

display(df_merged_pd.head())
display(df_merged_pd.shape)
display(df_merged_pd.dtypes)

display(f"Loading data took {time.time() - t1:.2f}s.")



In [ ]:
t2 = time.time()

cols_to_scale = ["lognbj", "log_ca"]

imputer = SimpleImputer(strategy = "median")
df_scaling_ready = imputer.fit_transform(df_merged_pd[cols_to_scale])


display(f"Imputing data took {time.time() - t2:.2f}s.")


t4 = time.time()

scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_scaling_ready)

display(f"Scaling data took {time.time() - t4:.2f}s.")

t3 = time.time()

encoder = OneHotEncoder(sparse_output = False)
cols_to_encod = ["Group_canal", "Max_RAT", "Typologie", "Typologie site", "region_cleaned"]

df_encoded = encoder.fit_transform(df_merged_pd[cols_to_encod])

display(f"Encoding data took {time.time() - t3:.2f}s.")

dt_faiss = np.concat([df_scaled, df_encoded], axis = 1).astype(np.float32)
display(dt_faiss)
display(dt_faiss.shape)
display(dt_faiss.dtype)


In [ ]:
%%time

# Plan centroids and user medoids

df_plan_index = df_merged.with_row_index()
plan_index = df_plan_index.group_by("Nom du forfait").agg(pl.col("index"))

mobile_plans = plan_index["Nom du forfait"].unique().to_list()
centroids = []
medoids = []

# Euclidean distance
index = faiss.IndexFlatL2(dt_faiss.shape[1])


for plan, indices in zip(plan_index["Nom du forfait"], plan_index["index"]):
    plan_user = dt_faiss[indices]
    plan_centroid = plan_user.mean(axis = 0, keepdims = True)
    centroids.append(plan_centroid[0, ])
    
    #df_plan = df_plan_index.filter(df_plan_index["index"].is_in(indices))
    
    index.reset()

    index.add(plan_user)
    
    score, ids = index.search(plan_centroid, k = 1)    
    medoids.append(plan_user[ids[0, 0]])


centroids_vectors = np.stack(centroids)
medoids_vectors = np.stack(medoids)

search_place_vectors = np.concat([centroids_vectors, medoids_vectors])

display(search_place_vectors)
display(search_place_vectors.shape)
display(search_place_vectors.dtype)
display(np.any(np.isnan(search_place_vectors)))



In [ ]:
%%time

tsne = TSNE(n_components = 2, perplexity  = 50, random_state = 4, n_jobs = -1)
coords = tsne.fit_transform(search_place_vectors)

plt.scatter(
    x = coords[ : , 0],
    y = coords[ : , 1]
)
plt.show()

In [ ]:
%%time

index = faiss.IndexFlatL2(search_place_vectors.shape[1])

index.add(search_place_vectors)
    
user_scores, ids = index.search(dt_faiss, k = 1)    


In [ ]:
%%time

# Heavily skewed. Heavily concetrated in dispersed points.
plt.hist(
    x = user_scores.flatten(),
    bins = 10000,
    range = (0, 6)
)
plt.ylim(0, 45000)
plt.show()

df_merged_pd["user_scores"] = user_scores
display(df_merged_pd["user_scores"].describe()) # mean ~= median: symmetrical data

In [ ]:
df_score_users = df_merged_pd.copy()

mean_user_score = df_score_users.groupby("msisdn").agg({"user_scores": "mean"})

IQR = mean_user_score.quantile(0.75) - mean_user_score.quantile(0.25)
#bound0 = (df_score_users["user_scores"].quantile(0.25) - 1.5 * IQR).iloc[0]
bound1 = (mean_user_score.quantile(0.75) + 1.5 * IQR).iloc[0]

df_score_users["user_scores"] = df_score_users.groupby("msisdn")["user_scores"].transform("mean")
df_underserved_users = df_score_users[df_score_users["user_scores"] > bound1]

display(df_underserved_users.head())
display(df_underserved_users.shape)
display(df_underserved_users.dtypes)
display(df_underserved_users.reset_index()["msisdn"].nunique())

In [ ]:
def get_mode(x):
    return x.mode()[0]

df_underserved_users_gp = df_underserved_users.groupby("msisdn").agg(
    {
    "lognbj": "mean",
    "log_ca": "mean",
    "Group_canal": get_mode,
    "Max_RAT": get_mode,
    "Typologie": get_mode,
    "Typologie site": get_mode,
    "region_cleaned": get_mode
    }
)

df_underserved_users_gp.head()

In [ ]:
t1 = time.time()

cols_to_scale = ["lognbj", "log_ca"]

df_num = imputer.transform(df_underserved_users_gp[cols_to_scale])


display(f"Imputing data took {time.time() - t1:.2f}s.")


t2 = time.time()

df_num_scaled = scaler.transform(df_num)

display(f"Scaling data took {time.time() - t2:.2f}s.")

t3 = time.time()

cols_to_encod = ["Group_canal", "Max_RAT", "Typologie", "Typologie site", "region_cleaned"]

df_cat = encoder.transform(df_underserved_users_gp[cols_to_encod])


display(f"Encoding data took {time.time() - t3:.2f}s.")

dt_faiss = np.concat([df_num_scaled, df_cat], axis = 1).astype(np.float32)
display(dt_faiss)
display(dt_faiss.shape)
display(dt_faiss.dtype)


In [ ]:
%%time

k_values = range(1, 100)
# SSE from centroid
inertias = []
for k in k_values:
    kmeans = KMeans(n_clusters = k, random_state = 4)
    kmeans.fit(dt_faiss)
    inertias.append(kmeans.inertia_)

plt.scatter(k_values, inertias, marker = "o")
plt.show()



In [ ]:
optimal_k = 25
kmeans = KMeans(n_clusters = optimal_k, random_state = 4)
kmeans.fit(dt_faiss)
df_underserved_users_gp["labels"] = kmeans.labels_


for label, df in df_underserved_users_gp.groupby("labels"):
    display(df.describe(include = "all"))
